In [1]:
import os
import re
import urllib3
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin, parse_qsl

In [2]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [3]:
BASE_DOWNLOAD_DIR = "./downloads"
os.makedirs(BASE_DOWNLOAD_DIR, exist_ok=True)

In [4]:
def sanitize_and_generate_dynamic_folder(raw_url: str) -> tuple:
    """
    Cleans trailing whitespaces and keeps the exact native parameter order 
    to prevent remote server mapping failures.
    """
    if not raw_url or not isinstance(raw_url, str):
        return None, None
        
    sanitized_url = raw_url.strip().replace('\n', '').replace('\r', '')
    parsed = urlparse(sanitized_url)
    netloc = parsed.netloc.lower()
    
    # Generate a completely clean folder name without risky characters
    query_dict = dict(parse_qsl(parsed.query, keep_blank_values=False))
    domain_clean = netloc.replace('www.', '')
    id_marker = query_dict.get('id') or query_dict.get('oga_service') or 'index'
    
    # Safe Windows/Linux folder slug conversion
    id_marker_clean = re.sub(r'[^a-zA-Z0-9_\-]', '_', str(id_marker))[:30].strip('_')
    final_folder_name = f"{domain_clean}_{id_marker_clean}"
    
    # Use normpath to ensure Windows backslashes scale cleanly
    unique_target_path = os.path.normpath(os.path.join(BASE_DOWNLOAD_DIR, final_folder_name))
    
    return sanitized_url, unique_target_path

In [9]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin, parse_qsl

def dynamic_session_extractor(target_url: str, session_folder_path: str):
    """
    Uses a full-stack anti-bot fingerprint to bypass false 404 security traps.
    Extracts binary files AND maps structured HTML text into a formatted layout.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "en-US,en;q=0.9",
        "Cache-Control": "max-age=0",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Ch-Ua": '"Not/A)Brand";v="8", "Chromium";v="126", "Google Chrome";v="126"',
        "Sec-Ch-Ua-Mobile": "?0",
        "Sec-Ch-Ua-Platform": '"Windows"',
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1"
    }
    
    try:
        with requests.get(target_url, headers=headers, stream=True, timeout=20, verify=False) as response:
            if response.status_code == 404:
                print("   ⚠ 404 Server Status: Target rejected or record missing.")
                return
            response.raise_for_status()
            
            content_type = response.headers.get('Content-Type', '').lower()
            content_disposition = response.headers.get('Content-Disposition', '')
            
            is_binary = any(t in content_type for t in ['pdf', 'spreadsheet', 'excel', 'word', 'octet-stream', 'zip'])
            is_forced = 'filename=' in content_disposition
            
            # --- SCENARIO A: The parent URL is a direct file pipeline stream ---
            if is_binary or is_forced:
                if 'text/html' in content_type: return
                
                if is_forced:
                    filename = re.findall(r'filename=["\']?(.*?)["\']?$', content_disposition)[0]
                else:
                    filename = os.path.basename(urlparse(target_url).path)
                    
                if not filename or '.' not in filename or filename.endswith('.php'):
                    ext = '.pdf' if 'pdf' in content_type else '.xlsx' if 'excel' in content_type else '.docx'
                    filename = f"document_download{ext}"
                
                os.makedirs(session_folder_path, exist_ok=True)
                with open(os.path.join(session_folder_path, filename), 'wb') as f:
                    for chunk in response.iter_content(chunk_size=16384): f.write(chunk)
                print(f"   ✓ Extracted direct asset file: {filename}")
                
            # --- SCENARIO B: HTML Page Node - Extract STRUCTURAL TEXT data and parse embedded links ---
            else:
                os.makedirs(session_folder_path, exist_ok=True)
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # 1. TEXT EXTRACTION FUNCTIONALITY FOR EMBEDDING PREPARATION
                for element in soup(["script", "style", "nav", "footer", "header", "aside"]):
                    element.decompose()
                
                structured_blocks = []
                tags_to_extract = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'ul', 'ol', 'table', 'form']
                
                all_elements = soup.find_all(tags_to_extract)
                
                for element in all_elements:
                    if element.find_parent(['table', 'form', 'ul', 'ol']):
                        continue
                        
                    # Maps Headings
                    if element.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
                        text = element.get_text(strip=True)
                        if text:
                            structured_blocks.append(f"\n{text}\n")
                    
                    # Map Paragraph Tags
                    elif element.name == 'p':
                        text = element.get_text(strip=True)
                        if text:
                            structured_blocks.append(text)
                    
                    # Maps Unordered and Ordered Lists into standard dash bullet items (-)
                    elif element.name in ['ul', 'ol']:
                        list_items = []
                        for li in element.find_all('li', recursive=False):
                            li_text = li.get_text(strip=True)
                            if li_text:
                                list_items.append(f"- {li_text}")
                        if list_items:
                            structured_blocks.append("\n" + "\n".join(list_items) + "\n")
                    
                    # UPDATED: Maps Tables into a key-value layout (Column: Row1, Row2, ...)
                    elif element.name == 'table':
                        headers_list = []
                        columns_data = {}
                        
                        # First, isolate header names safely
                        thead = element.find('thead')
                        header_source = thead if thead else element
                        first_tr = header_source.find('tr')
                        
                        if first_tr:
                            for th in first_tr.find_all(['th', 'td']):
                                header_text = th.get_text(strip=True)
                                headers_list.append(header_text)
                                columns_data[header_text] = []
                        
                        # Process all remaining body data rows
                        tbody = element.find('tbody')
                        row_source = tbody if tbody else element
                        all_trs = row_source.find_all('tr')
                        
                        # If headers were inside the loop selection, skip the first row
                        start_idx = 1 if not thead and first_tr in all_trs else 0
                        
                        for tr in all_trs[start_idx:]:
                            for idx, td in enumerate(tr.find_all(['td', 'th'])):
                                if idx < len(headers_list):
                                    # Handle layout breaks inside cells seamlessly
                                    cell_strings = [s for s in td.stripped_strings]
                                    cell_text = " ".join(cell_strings) if cell_strings else "-"
                                    
                                    current_header = headers_list[idx]
                                    columns_data[current_header].append(cell_text)
                        
                        # Build the text format output block
                        table_output = []
                        for header in headers_list:
                            rows_combined = ", ".join(columns_data[header])
                            table_output.append(f"{header}: {rows_combined}")
                            
                        if table_output:
                            structured_blocks.append("\n" + "\n".join(table_output) + "\n")
                    
                    # Map Forms systematically
                    elif element.name == 'form':
                        form_elements = []
                        for field in element.find_all(['label', 'input', 'select', 'textarea']):
                            if field.name == 'label':
                                form_elements.append(f"Label: {field.get_text(strip=True)}")
                            elif field.name in ['input', 'select', 'textarea']:
                                field_type = field.get('type', field.name)
                                field_name = field.get('name', 'unnamed')
                                form_elements.append(f"  -> [{field_type.upper()}] Name: {field_name}")
                        if form_elements:
                            structured_blocks.append("\n[FORM FIELD LAYOUT]\n" + "\n".join(form_elements) + "\n[FORM END]\n")

                final_page_text = "\n".join(structured_blocks).strip()
                
                # Save plain text payload to the dynamic folder location
                text_filename = "scraped_content.txt"
                text_filepath = os.path.join(session_folder_path, text_filename)
                with open(text_filepath, "w", encoding="utf-8") as text_file:
                    text_file.write(f"Source URL: {target_url}\n")
                    text_file.write("="*80 + "\n\n")
                    text_file.write(final_page_text)
                print(f"   ✓ Extracted page content text -> Saved: {text_filename}")
                
                # 2. EMBEDDED FILE ATTACHMENT SCRAPING LOOP
                anchors = soup.find_all('a', href=True)
                doc_extensions = re.compile(r'\.(pdf|xlsx|xls|docx|doc)(\?|$)', re.IGNORECASE)
                keyword_triggers = ['download', 'attachment', 'view-service', 'viewfile', 'form', 'document']
                
                valid_links = set()
                for link in anchors:
                    href = link['href']
                    text = link.get_text(strip=True).lower()
                    
                    if doc_extensions.search(href) or any(k in href.lower() or k in text for k in keyword_triggers):
                        if not any(term in href.lower() for term in ['login', 'logout', 'javascript:', 'facebook', 'twitter']):
                            valid_links.add(urljoin(target_url, href))
                
                if not valid_links:
                    print("   -> Status: Text captured, but no binary download links were found.")
                    return
                    
                for asset_route in valid_links:
                    try:
                        with requests.get(asset_route, headers=headers, stream=True, timeout=15, verify=False) as sub_res:
                            sub_type = sub_res.headers.get('Content-Type', '').lower()
                            if 'text/html' in sub_type or not sub_res.ok: continue
                            
                            sub_disp = sub_res.headers.get('Content-Disposition', '')
                            if 'filename=' in sub_disp:
                                sub_filename = re.findall(r'filename=["\']?(.*?)["\']?$', sub_disp)[0]
                            else:
                                sub_filename = os.path.basename(urlparse(asset_route).path)
                                
                            if not sub_filename or '.' not in sub_filename or sub_filename.endswith('.php'):
                                sub_query = dict(parse_qsl(urlparse(asset_route).query))
                                asset_id = sub_query.get('id') or 'file_record'
                                ext = '.pdf' if 'pdf' in sub_type else '.xlsx' if 'excel' in sub_type else '.docx'
                                sub_filename = f"extracted_asset_{asset_id}{ext}"
                            
                            with open(os.path.join(session_folder_path, sub_filename), 'wb') as f_sub:
                                for chunk in sub_res.iter_content(chunk_size=16384): f_sub.write(chunk)
                            print(f"      ✓ Extracted Asset Captured: {sub_filename}")
                    except Exception: pass
    except Exception as e: print(f"   ✗ Connection Interrupted: {e}")

In [10]:
if __name__ == "__main__":
    print("=== INITIALIZING SERVER-ALIGNMENT BATCH PIPELINE ===")
    
    DYNAMIC_TEST_POOL = [
        "https://tipp.gov.pk/index.php?r=SearchForms/view&id=19",
        "https://www.psw.gov.pk/view-service-catalouge?oga_service=DRAP%20-%20Drug%20Regulatory%20Authority%20of%20Pakistan",
        "https://tipp.gov.pk/index.php?r=SearchForms/view&id=38",
        "https://tipp.gov.pk/index.php?r=searchProcedure/view1&id=30"
    ]
    
    print(f"Queue Size: {len(DYNAMIC_TEST_POOL)} locations locked into memory.\n" + "="*60)
    
    for idx, raw_url in enumerate(DYNAMIC_TEST_POOL, start=1):
        clean_url, session_workspace = sanitize_and_generate_dynamic_folder(raw_url)
        if not clean_url: continue
        
        print(f"\n[{idx}/{len(DYNAMIC_TEST_POOL)}] Initializing Target Container Workflow")
        print(f"   ↳ Target URL: {clean_url}")
        print(f"   ↳ Destination Path: {session_workspace}")
        
        dynamic_session_extractor(clean_url, session_workspace)
        
    print("\n" + "="*60 + "\n=== SYSTEM PIPELINE PROCESSING COMPLETE ===")

=== INITIALIZING SERVER-ALIGNMENT BATCH PIPELINE ===
Queue Size: 4 locations locked into memory.

[1/4] Initializing Target Container Workflow
   ↳ Target URL: https://tipp.gov.pk/index.php?r=SearchForms/view&id=19
   ↳ Destination Path: downloads\tipp.gov.pk_19
   ✓ Extracted page content text -> Saved: scraped_content.txt
      ✓ Extracted Asset Captured: form_10.pdf

[2/4] Initializing Target Container Workflow
   ↳ Target URL: https://www.psw.gov.pk/view-service-catalouge?oga_service=DRAP%20-%20Drug%20Regulatory%20Authority%20of%20Pakistan
   ↳ Destination Path: downloads\psw.gov.pk_DRAP_-_Drug_Regulatory_Authori
   ✓ Extracted page content text -> Saved: scraped_content.txt

[3/4] Initializing Target Container Workflow
   ↳ Target URL: https://tipp.gov.pk/index.php?r=SearchForms/view&id=38
   ↳ Destination Path: downloads\tipp.gov.pk_38
   ✓ Extracted page content text -> Saved: scraped_content.txt
      ✓ Extracted Asset Captured: Format of Export Registration Certificate.docx

[